<br/>

<div align="center">
<span style="font-size: 2.5em;">XENON Preprocessing Step 2: Merge Fuse Output</span>
<br/>
<span style="font-size: 1.2em; color: gray;">Combine WimPyDD spectra with fuse-simulated (cS1, cS2) detector response</span>
</div>

## Overview

The fuse simulation outputs (cS1, cS2) pairs for each recoil event in CSV format. This notebook merges those detector observables back into the original `.pt` files, adding:

- **cs1cs2**: Corrected S1/S2 signals for each event
- **xyz**: Event positions (x, y, z) in the detector

This creates a complete dataset for training neural density estimators on realistic XENON observables.

**Important**: Verify `torch.save()` is uncommented before running to persist the merged data.

## Setup and Imports

In [ ]:
# =============================
# IMPORTS AND PATH SETTINGS
# =============================

import os
import numpy as np
import pandas as pd
import torch

desired_root_name = "xenon-sbi"  
while os.path.basename(os.getcwd()) != desired_root_name:
    os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

## Configuration

In [14]:
# Dataset parameters
n = 300000
halo = "x2"
DATATAG = "low"

# File paths
PT_FILE = os.path.join("data", "datasets", "wimpy", f"{halo}", f"wimpy_n{n}_{DATATAG}_{halo}.pt")
CSV_FILE = os.path.join("data", "datasets", "xenon", "s1s2", "csv", f"s1s2_n{n}_{halo}.csv")
OUT_FILE = os.path.join("data", "datasets", "xenon", "s1s2", "pt", f"s1s2_n{n}_{halo}.pt")

print(f"Input PT:  {PT_FILE}")
print(f"Input CSV: {CSV_FILE}")
print(f"Output:    {OUT_FILE}")

Input PT:  data\datasets\wimpy\x2\wimpy_n300000_low_x2.pt
Input CSV: data\datasets\xenon\s1s2\csv\s1s2_n300000_x2.csv
Output:    data\datasets\xenon\s1s2\pt\s1s2_n300000_x2.pt


## Load Data

In [15]:
# Load original WimPyDD dataset
data = torch.load(PT_FILE, weights_only=False)
events_list = data["events"]
print(f"Loaded {len(events_list)} parameter points from PT file")

# Load fuse simulation results
df = pd.read_csv(CSV_FILE)
print(f"Loaded {len(df)} total events from CSV")
print(f"Columns: {df.columns.tolist()}")

Loaded 300000 parameter points from PT file
Loaded 4288158 total events from CSV
Columns: ['ed', 'cs1', 'cs2', 'micro_endtime', 'event_endtime', 'time_diff_ns', 'event_index', 'chunk_id', 'spectrum_id', 'local_id', 'xp', 'yp', 'zp']


## Merge Event-Level Data

Match each parameter point's events with corresponding (cS1, cS2, x, y, z) values from the fuse output.

**Note**: Some events may contain NaN values, e.g. resulting from detector tresholds. These will be filtered out during the training preprocessing pipeline.

In [16]:
cs1cs2_list = []
xyz_list = []
index = 0

for events in events_list:
    n_events = len(events)
    
    if n_events == 0:
        # No events for this parameter point
        cs1cs2_list.append(np.empty((0, 2)))
        xyz_list.append(np.empty((0, 3)))
    else:
        # Extract corresponding rows from CSV
        tmp = df.iloc[index : index + n_events]
        cs1cs2 = np.column_stack((tmp["cs1"].to_numpy(), tmp["cs2"].to_numpy()))
        xyz = np.column_stack((tmp["xp"].to_numpy(), tmp["yp"].to_numpy(), tmp["zp"].to_numpy()))
        cs1cs2_list.append(cs1cs2)
        xyz_list.append(xyz)
    
    index += n_events

print(f"Merged {len(cs1cs2_list)} parameter points")

Merged 300000 parameter points


## Save Combined Dataset

In [17]:
# Add new fields to dataset
data["cs1cs2"] = cs1cs2_list
data["xyz"] = xyz_list

# Save merged dataset (IMPORTANT: uncomment to persist!)
torch.save(data, OUT_FILE)
print(f"✓ Combined dataset saved to {OUT_FILE}")

print("⚠ Dataset NOT saved (torch.save is commented out)")
print("Uncomment the torch.save line above to persist merged data")

✓ Combined dataset saved to data\datasets\xenon\s1s2\pt\s1s2_n300000_x2.pt
⚠ Dataset NOT saved (torch.save is commented out)
Uncomment the torch.save line above to persist merged data


## Validation

Verify the merged dataset contains all expected fields with consistent dimensions.

In [18]:
# Load saved file (only works if torch.save was executed above)
# data = torch.load(OUT_FILE, weights_only=False)

# For now, validate in-memory data
events_list = data["events"]
s1s2_list = data["cs1cs2"]
xyz_list = data["xyz"]

print("Dataset keys:", list(data.keys()))
print(f"\nNumber of parameter points: {len(events_list)}")

# Show example from parameter point #3
idx = 3
print(f"\n--- Example: Parameter point {idx} ---")
print(f"Events (energies):     {events_list[idx]}")
print(f"Detector signals (cS1, cS2):\n{s1s2_list[idx]}")
print(f"Positions (x, y, z):\n{xyz_list[idx]}")

Dataset keys: ['theta', 'features', 'events', 'logcp_range', 'logm_range', 'n_train', 'top_k', 'halo_option', 'shmpp_file', 'mc_config', 'cs1cs2', 'xyz']

Number of parameter points: 300000

--- Example: Parameter point 3 ---
Events (energies):     [ 9.04401    5.7597995 14.003544  10.793712   4.8926406  9.178795
  2.5150414  3.4351914  1.2939916  4.173923  11.650131  16.016714
  3.6915302  1.8159235  2.073273   5.797579   7.0323     5.248837
  7.258482  14.480404   6.8855386  3.5145643  7.594828   7.169603
  4.6575356 14.633931   2.0050986  2.5106971  2.3676643 11.379852
  1.2291846  1.2812022 12.929324   5.563021   4.428117  23.368505
  1.3715186  4.680499   2.7319405  4.4281864  7.412304  12.571138
  1.1442584 19.162224  16.226105   9.235786   5.1554437  2.564215
  4.150336   4.2792945  4.771659   5.947967   5.840521  12.74969
  3.0021367  4.6714463  3.4888473  2.3852465 17.42419    2.0952744
  5.5585594 10.320327   7.3105245 14.214168   7.345208   3.2747514
  6.756251   3.6256673  